# Baseline — ANN (MovieLens)
OHE + multi-hot · Plain MLP · No embeddings
**Saves to**: `baseline_ANN\`

## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Load data
STEP 3 — Numerical preprocessing
STEP 4 — Build OHE feature matrix (dense)
STEP 5 — PyTorch Dataset & DataLoader
STEP 6 — Model definition (plain MLP)
STEP 7 — Train
STEP 8 — Evaluate
STEP 9 — Save results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data      import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute        import SimpleImputer
from sklearn.metrics       import mean_squared_error, mean_absolute_error
from scipy.sparse          import csr_matrix, hstack

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device: {DEVICE}')

In [ ]:
DATA_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\data"
OUT_DIR  = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\baseline_ANN"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print(f"Saving results to: {{OUT_DIR}}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(DATA_DIR + '\\train.csv')
df_val   = pd.read_csv(DATA_DIR + '\\val.csv')
df_test  = pd.read_csv(DATA_DIR + '\\test.csv')

TARGET    = 'rating'
CAT_GENRE = 'genres'
CAT_LANG  = 'original_language'
NUM_COLS  = [
    'imdbRating', 'imdbVotes', 'tmdbRating', 'tmdbVotes',
    'popularity', 'budget', 'runtime', 'revenue',
    'tag_count',  'avg_relevance', 'max_relevance'
]
y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values
print(f"Train: {{df_train.shape}}  Val: {{df_val.shape}}  Test: {{df_test.shape}}")


---
## Step 3 — Numerical Preprocessing

In [ ]:
# Tag features: missing = no tag activity → 0
for df_ in [df_train, df_val, df_test]:
    df_[['tag_count', 'avg_relevance', 'max_relevance']] = \
        df_[['tag_count', 'avg_relevance', 'max_relevance']].fillna(0)


In [ ]:
# Median imputation — fit on train only
imputer = SimpleImputer(strategy='median')
df_train[NUM_COLS] = imputer.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = imputer.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = imputer.transform(df_test[NUM_COLS])


In [ ]:
# log1p transform
for df_ in [df_train, df_val, df_test]:
    df_[NUM_COLS] = np.log1p(df_[NUM_COLS].clip(lower=0))


In [ ]:
# StandardScaler — fit on train only
scaler = StandardScaler()
df_train[NUM_COLS] = scaler.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = scaler.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = scaler.transform(df_test[NUM_COLS])
print("Numerical preprocessing done.")


---
## Step 4 — Build OHE Feature Matrix (Dense)
Same OHE + multi-hot as classical models. Converted to dense for PyTorch.

In [ ]:
# Multi-hot encode genres — fit on train, reindex val/test
genres_tr = df_train[CAT_GENRE].fillna('Unknown').str.get_dummies('|')
genre_cols = genres_tr.columns
genres_v  = df_val[CAT_GENRE].fillna('Unknown').str.get_dummies('|').reindex(columns=genre_cols, fill_value=0)
genres_te = df_test[CAT_GENRE].fillna('Unknown').str.get_dummies('|').reindex(columns=genre_cols, fill_value=0)
print(f"Genre columns: {{len(genre_cols)}}")


In [ ]:
# OHE language — fit on train only
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
lang_tr = ohe.fit_transform(df_train[[CAT_LANG]].fillna('unknown'))
lang_v  = ohe.transform(df_val[[CAT_LANG]].fillna('unknown'))
lang_te = ohe.transform(df_test[[CAT_LANG]].fillna('unknown'))
print(f"Language columns: {{len(ohe.categories_[0])}}")


In [ ]:
# Stack numeric + language + genre into sparse matrix
X_train = hstack([csr_matrix(df_train[NUM_COLS].values), lang_tr, csr_matrix(genres_tr.values)])
X_val   = hstack([csr_matrix(df_val[NUM_COLS].values),   lang_v,  csr_matrix(genres_v.values)])
X_test  = hstack([csr_matrix(df_test[NUM_COLS].values),  lang_te, csr_matrix(genres_te.values)])
feat_names = NUM_COLS + ohe.get_feature_names_out([CAT_LANG]).tolist() + list(genre_cols)
print(f"Feature matrix — train: {{X_train.shape}}")


In [ ]:
X_train_dense = X_train.toarray().astype(np.float32)
X_val_dense   = X_val.toarray().astype(np.float32)
X_test_dense  = X_test.toarray().astype(np.float32)
print(f'Dense feature matrix — train: {X_train_dense.shape}')

---
## Step 5 — PyTorch Dataset & DataLoader

In [ ]:
class RatingsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 512
train_loader = DataLoader(RatingsDataset(X_train_dense, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(RatingsDataset(X_val_dense,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(RatingsDataset(X_test_dense,  y_test),  batch_size=BATCH_SIZE, shuffle=False)
X_b, y_b = next(iter(train_loader))
print(f'Batch shapes — X: {X_b.shape}  y: {y_b.shape}')

---
## Step 6 — Model Definition (Plain MLP)
No embedding layers — input is raw OHE feature vector.
Architecture: `input → 128 → ReLU → 64 → ReLU → 1`

In [ ]:
class PlainMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x).squeeze(1)

model = PlainMLP(X_train_dense.shape[1]).to(DEVICE)
print(model)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

---
## Step 7 — Train

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            preds = model(X_b)
            loss  = criterion(preds, y_b)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(y_b.cpu().numpy())
    p = np.array(all_preds); l = np.array(all_labels)
    return np.sqrt(mean_squared_error(l,p)), mean_absolute_error(l,p)

In [ ]:
NUM_EPOCHS = 20
criterion  = nn.MSELoss()
optimizer  = optim.Adam(model.parameters(), lr=0.001)
history    = {'train_rmse': [], 'val_rmse': []}
t0 = time.perf_counter()

for epoch in range(NUM_EPOCHS):
    tr_rmse, _  = run_epoch(model, train_loader, criterion, optimizer)
    val_rmse, _ = run_epoch(model, val_loader,   criterion)
    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(val_rmse)
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:>2}/{NUM_EPOCHS}  Train RMSE: {tr_rmse:.4f}  Val RMSE: {val_rmse:.4f}')

train_time = time.perf_counter() - t0
print(f'Total training time: {train_time:.1f}s')

In [ ]:
plt.figure(figsize=(8,3))
plt.plot(history['train_rmse'], label='Train RMSE')
plt.plot(history['val_rmse'],   label='Val RMSE')
plt.xlabel('Epoch'); plt.ylabel('RMSE')
plt.title('ANN Baseline — Learning Curves')
plt.legend(); plt.tight_layout(); plt.show()

---
## Step 8 — Evaluate

In [ ]:
tr_rmse,  tr_mae  = run_epoch(model, train_loader, criterion)
val_rmse, val_mae = run_epoch(model, val_loader,   criterion)
te_rmse,  te_mae  = run_epoch(model, test_loader,  criterion)

print('=' * 50)
print('ANN — Baseline Results')
print('=' * 50)
print(f'  Train      RMSE: {tr_rmse:.4f}   MAE: {tr_mae:.4f}')
print(f'  Validation RMSE: {val_rmse:.4f}   MAE: {val_mae:.4f}')
print(f'  Test       RMSE: {te_rmse:.4f}   MAE: {te_mae:.4f}')
print(f'  Train time : {train_time:.1f}s')
print(f'  Train-Test gap : {te_rmse - tr_rmse:.4f}')

---
## Step 9 — Save Results

In [ ]:
result = {
    'model'        : 'ANN (Baseline)',
    'train_rmse'   : round(tr_rmse,  4),
    'val_rmse'     : round(val_rmse, 4),
    'test_rmse'    : round(te_rmse,  4),
    'train_mae'    : round(tr_mae,   4),
    'val_mae'      : round(val_mae,  4),
    'test_mae'     : round(te_mae,   4),
    'train_time_s' : round(train_time, 2),
}
pd.DataFrame([result]).to_csv(os.path.join(OUT_DIR, 'ann_results.csv'), index=False)
with open(os.path.join(OUT_DIR, 'ann_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(f"Results saved → {OUT_DIR}")
print(f"Test RMSE: {te_rmse:.4f}  Test MAE: {te_mae:.4f}")
